# Tutorial: Feature Engineering

> **Type:** Tutorial — follow along step by step. Your only job is to make it work. Don't skip cells.
>
> **Time:** ~1.5-2 hours
> **Prerequisites:** Module 07 (data cleaning), Module 09 (EDA), Module 13 (cross-validation)
> **What you'll build:** A leakage-safe `ColumnTransformer` pipeline that turns a raw e-commerce orders dataset into an ML-ready feature matrix — with encoded categoricals, scaled numerics, datetime features, interaction terms, and binned variables.

---

## What We're Building

You'll work with a synthetic e-commerce orders dataset containing numeric columns (order value, customer age), categorical columns (region, payment method — one low cardinality, one high cardinality), and a datetime column (order timestamp). By the end you'll have a single reusable `sklearn.pipeline.Pipeline` that takes raw rows in and produces a clean feature matrix out, fit only on training data.

Don't worry yet about understanding every line. That's what `explanation.md` is for. Here, just follow the steps and watch it work.

## Step 0: Setup

Install and import everything we need.

In [ ]:
# !pip install numpy pandas scikit-learn matplotlib category_encoders

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler,
    RobustScaler, PolynomialFeatures, KBinsDiscretizer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 11
print('Setup complete.')

## Step 1: Build a Synthetic Orders Dataset

We simulate an e-commerce orders table — the kind of raw data you'd pull from a production database. It has a mix of numeric, low-cardinality categorical, high-cardinality categorical, and datetime columns, and the target (`order_value`) genuinely depends on interactions between columns so that feature engineering has a visible payoff later.

In [ ]:
n = 4000

regions = np.random.choice(['North', 'South', 'East', 'West'], size=n, p=[0.3, 0.25, 0.25, 0.2])
customer_id = np.random.randint(1000, 1000 + 400, size=n)  # high-cardinality: 400 unique customers
payment_method = np.random.choice(['credit_card', 'paypal', 'gift_card', 'bank_transfer'], size=n)
customer_age = np.random.normal(40, 12, size=n).clip(18, 80)

start = pd.Timestamp('2024-01-01')
order_timestamp = start + pd.to_timedelta(np.random.randint(0, 365 * 24 * 60, size=n), unit='m')

items_in_cart = np.random.poisson(3, size=n) + 1

# Region and payment method genuinely shift spend — this is the signal we want features to expose
region_premium = pd.Series(regions).map({'North': 1.15, 'South': 0.95, 'East': 1.05, 'West': 0.90}).values
weekend_boost = pd.Series(order_timestamp).dt.dayofweek.isin([5, 6]).astype(int).values * 8

order_value = (
    15 * items_in_cart * region_premium
    + 0.3 * customer_age
    + weekend_boost
    + np.random.normal(0, 10, size=n)
).clip(5, None)

df = pd.DataFrame({
    'customer_id': customer_id,
    'region': regions,
    'payment_method': payment_method,
    'customer_age': customer_age,
    'items_in_cart': items_in_cart,
    'order_timestamp': order_timestamp,
    'order_value': order_value,
})

df.head()

> **Check:** You should see 4000 rows and 7 columns, with `order_timestamp` as a real datetime and `region`/`payment_method` as strings. If `order_timestamp` printed as an object/string instead of a datetime, re-run the cell above — `pd.to_timedelta` needs the `start` Timestamp to build proper datetimes.

## Step 2: Split Before You Engineer Anything

This is the single most important step in the whole notebook. Every encoder and scaler we fit below must see **only the training set**. If you fit on the full dataset first, the target-encoding statistics and scaler means will have peeked at the test set — that's leakage, and it will make your validation metrics lie to you.

In [ ]:
X = df.drop(columns=['order_value'])
y = df['order_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Step 3: Datetime Feature Extraction

`order_timestamp` on its own is useless to a model — it's just a large, arbitrary number. The signal is *inside* it: day of week, hour, whether it's a weekend. We also cyclically encode `hour` and `day_of_week` with sin/cos so that, e.g., hour 23 and hour 0 are recognised as adjacent instead of maximally far apart.

In [ ]:
def add_datetime_features(frame):
    frame = frame.copy()
    ts = frame['order_timestamp']
    frame['hour'] = ts.dt.hour
    frame['day_of_week'] = ts.dt.dayofweek       # 0 = Monday
    frame['month'] = ts.dt.month
    frame['is_weekend'] = ts.dt.dayofweek.isin([5, 6]).astype(int)

    # Cyclical encoding — makes 23:00 and 00:00 numerically adjacent
    frame['hour_sin'] = np.sin(2 * np.pi * frame['hour'] / 24)
    frame['hour_cos'] = np.cos(2 * np.pi * frame['hour'] / 24)
    frame['dow_sin'] = np.sin(2 * np.pi * frame['day_of_week'] / 7)
    frame['dow_cos'] = np.cos(2 * np.pi * frame['day_of_week'] / 7)

    return frame.drop(columns=['order_timestamp'])

X_train_dt = add_datetime_features(X_train)
X_test_dt = add_datetime_features(X_test)
X_train_dt.head()

> **What just happened?** `order_timestamp` became eight numeric columns. `hour_sin`/`hour_cos` together encode the hour as a point on a circle — no leakage risk here since this is a pure function of the timestamp itself, not of the target.

## Step 4: Visualise Why Cyclical Encoding Matters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(range(24), range(24), c=range(24), cmap='twilight')
axes[0].set_title('Raw hour (0-23)\nhour 23 and hour 0 look maximally far apart')
axes[0].set_xlabel('hour')
axes[0].set_ylabel('hour')

hours = np.arange(24)
hs, hc = np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24)
axes[1].scatter(hs, hc, c=hours, cmap='twilight')
for h in [0, 6, 12, 18, 23]:
    axes[1].annotate(str(h), (hs[h], hc[h]))
axes[1].set_title('sin/cos encoded hour\nhour 23 and hour 0 are now neighbours')
axes[1].set_xlabel('hour_sin')
axes[1].set_ylabel('hour_cos')

plt.tight_layout()
plt.show()

## Step 5: Encoding Categorical Variables

We have two categoricals with very different cardinality:

- `region`: 4 unique values → **one-hot encode**. Cheap, no ordering implied, model can learn a separate weight per region.
- `customer_id`: ~400 unique values → one-hot encoding would add 400 columns for a single field. Instead we use **frequency encoding** (safe, no leakage) — later, `howto.md` also covers target encoding with proper cross-validation for cases where frequency isn't informative enough.

`payment_method` (4 values, no natural order) also gets one-hot encoded.

In [ ]:
# Frequency encoding for high-cardinality customer_id — computed on TRAIN ONLY
customer_freq = X_train_dt['customer_id'].value_counts(normalize=True)

def add_frequency_encoding(frame, freq_map):
    frame = frame.copy()
    # unseen customers in test set get frequency 0 (never seen in training)
    frame['customer_freq'] = frame['customer_id'].map(freq_map).fillna(0.0)
    return frame.drop(columns=['customer_id'])

X_train_freq = add_frequency_encoding(X_train_dt, customer_freq)
X_test_freq = add_frequency_encoding(X_test_dt, customer_freq)  # reuses TRAIN frequencies

X_train_freq[['customer_freq']].describe()

> **What just happened?** `customer_freq` replaces 400 raw IDs with a single number: how often that customer appears in the *training* set. Notice `X_test_freq` reused `customer_freq` computed from train — this is the leakage-safe pattern. A customer who only appears in the test set correctly gets frequency 0.

## Step 6: Interaction and Polynomial Features

`items_in_cart` and `region` interact: the same cart size is worth more in a premium region. We can't multiply a category directly, but after one-hot encoding we can create interaction terms, or let `PolynomialFeatures` generate all pairwise products of the numeric columns automatically.

In [ ]:
numeric_cols = ['customer_age', 'items_in_cart']

poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
poly_train = poly.fit_transform(X_train_freq[numeric_cols])
poly_feature_names = poly.get_feature_names_out(numeric_cols)

pd.DataFrame(poly_train, columns=poly_feature_names).head()

> **What just happened?** `PolynomialFeatures(degree=2)` turned 2 columns into 5: the originals, their squares, and their product (`customer_age items_in_cart`). For a linear model, that product term is the only way to capture "older customers with big carts spend disproportionately more" — a tree-based model would find this split on its own, but a linear model needs it spelled out.

## Step 7: Scaling Numeric Features

Scaling matters for linear models and distance-based models, not for trees. We compare three scalers on `customer_age`:

- `StandardScaler` — mean 0, std 1. Default choice.
- `MinMaxScaler` — squashes to [0, 1]. Use when you need bounded input (e.g. neural nets).
- `RobustScaler` — uses median/IQR. Use when outliers would distort mean/std.

In [ ]:
age = X_train_freq[['customer_age']]

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, scaler) in zip(axes, scalers.items()):
    scaled = scaler.fit_transform(age)
    ax.hist(scaled, bins=30, color='#4E79A7')
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Step 8: Binning a Continuous Variable

Sometimes the *bucket* a value falls into matters more than its exact value — marketing segments customers into age bands rather than modelling age continuously. `KBinsDiscretizer` automates this.

In [ ]:
binner = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
age_binned = binner.fit_transform(X_train_freq[['customer_age']])

pd.Series(age_binned.ravel()).value_counts().sort_index()

> **What just happened?** `strategy='quantile'` chose bin edges so each of the 4 bins has roughly the same number of customers — useful when you want balanced groups rather than fixed-width age ranges.

## Step 9: Assemble Everything into One Leakage-Safe Pipeline

Doing each transform by hand was useful for seeing what's happening, but it's error-prone and easy to leak. `ColumnTransformer` + `Pipeline` apply every transform in one `fit`/`transform` call, fit only on training data, and can be dropped straight into cross-validation or a SageMaker inference container.

In [ ]:
numeric_features = ['customer_age', 'items_in_cart', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
categorical_features = ['region', 'payment_method']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_features),
])

full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=RANDOM_SEED)),
])

# Rebuild train/test with the columns the pipeline expects (drop raw datetime + customer_id,
# keep engineered datetime columns from Step 3, reuse the frequency-encoded frame's columns)
X_train_final = X_train_freq[numeric_features + categorical_features]
X_test_final = X_test_freq[numeric_features + categorical_features]

full_pipeline.fit(X_train_final, y_train)
print(f"Test R^2: {full_pipeline.score(X_test_final, y_test):.3f}")

> **What just happened?** `ColumnTransformer` routed numeric columns through imputation+scaling and categorical columns through imputation+one-hot, all inside a single `Pipeline`. Calling `.fit()` fits every step on `X_train_final` only; `.score()` on the held-out test set gives an honest, leakage-free estimate of performance.

## Step 10: Cross-Validate the Full Pipeline

In [ ]:
scores = cross_val_score(full_pipeline, X_train_final, y_train, cv=5, scoring='r2')
print(f"CV R^2: {scores.mean():.3f} +/- {scores.std():.3f}")

> **What just happened?** Because `full_pipeline` includes the preprocessing steps, `cross_val_score` refits the scaler and encoder fresh on each fold's training split — this is the correct way to cross-validate feature engineering. If you had scaled `X_train_final` once before calling `cross_val_score`, each fold's validation data would have leaked into the scaler's mean/std.

## What You Built

You turned a raw orders table — categoricals, a high-cardinality ID column, and a datetime — into a leakage-safe, cross-validation-ready feature matrix, and packaged the whole transformation into one reusable `Pipeline` that a Ridge regression can consume end to end.

## What's Next

- Read `explanation.md` to understand *why* each transformation works and when to reach for it.
- Open `howto.md` when you need to repeat one of these recipes on your own data.
- Keep `reference.md` open as a cheat sheet for encoder/scaler signatures.
- **Extension challenge:** Add a target-encoded version of `customer_id` (see `howto.md` for the leakage-safe cross-validation pattern) and compare CV R² against the frequency-encoded version used here.

---

**AWS SageMaker version:** The same `ColumnTransformer` pipeline can be wrapped in a SageMaker Scikit-learn Processing job, or reimplemented as a SageMaker Data Wrangler flow for a no-code equivalent — see `howto.md` for the SageMaker Feature Store pattern.